# Creator Revenue Prediction — Colab pipeline
Notebook chạy pipeline production từ GitHub: data cleaning, feature engineering, model benchmark, SHAP và dự báo doanh thu từng KOL.

In [ ]:
# 1. Cài đặt môi trường và lấy source code từ GitHub
import os, sys, subprocess
REPO_URL = 'https://github.com/kieu-collab/CreatorRevenuePrediction.git'
PROJECT_DIR = '/content/CreatorRevenuePrediction'
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=False)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project directory:', os.getcwd())

In [ ]:
# 2. Đọc, làm sạch dữ liệu và tạo marketing features
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.data_processing import load_and_clean, validate_dataset
from src.feature_engineering import build_features
DATA_PATH = Path('data/creator_campaign.csv')
df = load_and_clean(DATA_PATH, require_target=True)
warnings = validate_dataset(df, require_target=True)
features = build_features(df)
print(f"Rows: {len(df):,} | Creators: {df['creator_id'].nunique():,}")
print('Validation warnings:', warnings)
display(df.head())

In [ ]:
# 3. Xem benchmark và các yếu tố ảnh hưởng
display(pd.read_csv('models/metrics.csv'))
display(pd.read_csv('models/feature_importance.csv').head(15))
display(pd.read_csv('models/shap_summary.csv').head(15))

In [ ]:
# 4. Train lại toàn bộ mô hình
from src.train_model import train_and_save
selected_models = {'Linear Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM', 'CatBoost', 'Neural Network'}
metadata = train_and_save(DATA_PATH, Path('models'), selected_models=selected_models)
print('Best model:', metadata['best_model'])
display(pd.read_csv('models/metrics.csv'))

# 5. Dự báo doanh thu cho từng KOL và tạo bảng xếp hạng
from src.batch_prediction import generate_oof_predictions
summary = generate_oof_predictions(DATA_PATH, Path('models/best_model.pkl'), Path('outputs'), n_splits=5)
ranking = pd.read_csv('outputs/kol_revenue_ranking.csv')
print(summary)
display(ranking.head(20))